In [1]:
import shutil
shutil.copytree(r"/kaggle/input/datasets/xiaokonglong80/yolo11-hand-gesture-recognition", r"/kaggle/working/yolo11-hand-gesture-recognition")

'/kaggle/working/yolo11-hand-gesture-recognition'

In [2]:
!pip install -e /kaggle/working/yolo11-hand-gesture-recognition

Obtaining file:///kaggle/working/yolo11-hand-gesture-recognition
ERROR: file:///kaggle/working/yolo11-hand-gesture-recognition does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [3]:
!pip uninstall -y ray

Found existing installation: ray 2.54.0
Uninstalling ray-2.54.0:
  Successfully uninstalled ray-2.54.0


In [4]:
%cd /kaggle/working/yolo11-hand-gesture-recognition/ultralytics/cfg/datasets

/kaggle/working/yolo11-hand-gesture-recognition/ultralytics/cfg/datasets


In [5]:
%%writefile GES.yaml
# Ultralytics YOLO 🚀, AGPL-3.0 license
# COCO128 dataset https://www.kaggle.com/datasets/ultralytics/coco128 (first 128 images from COCO train2017) by Ultralytics
# Documentation: https://docs.ultralytics.com/datasets/detect/coco/
# Example usage: yolo train data=coco128.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── coco128  ← downloads here (7 MB)

# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: /kaggle/working/yolo11-hand-gesture-recognition/ges_dataset # dataset root dir
train: train/images  # train images (relative to 'path')
val: val/images  # val images (relative to 'path')
test: test/images  # test images (relative to 'path') 

# Classes
nc: 7  # number of classes
names:
  0: forward
  1: backward
  2: left
  3: right
  4: rotate_left
  5: rotate_right
  6: stop


Writing GES.yaml


In [6]:
%cd /kaggle/working/yolo11-hand-gesture-recognition

/kaggle/working/yolo11-hand-gesture-recognition


In [7]:
%%writefile train.py
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# 强制将当前目录加入系统路径，确保 DDP 子进程能找到 ultralytics
sys.path.append(os.getcwd())

from ultralytics import YOLO

if __name__ == '__main__':
    os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"

    os.environ["WANDB_API_KEY"] = "f4006571baebc40f3fe00fe509979ec34ff6455c"
    os.environ["WANDB_MODE"] = "online"
    model = YOLO('yolo11n.yaml') 
    model.train(data=r'GES.yaml',
                cache='ram',
                imgsz=640,
                epochs=500,
                batch=128,          
                device=[0, 1],      
                project='runs/train',
                name='exp',
                cfg='config.yaml',
                workers=4           # DDP 模式下 workers 不要设太高，防止内存溢出
                )

Overwriting train.py


In [8]:
# !python train.py
!python -m torch.distributed.run --nproc_per_node 2 ./train.py


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
New https://pypi.org/project/ultralytics/8.4.33 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.55 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                        CUDA:1 (Tesla T4, 14913MiB)
100%|████████████████████████████████████████| 755k/755k [00:00<00:00, 16.1MB/s]
100%|████████████████████████████████████████| 755k/755k [00:00<00:00, 19.1MB/s]
E0000 00:00:1775013207.476999     116 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775013207.540205     116 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempti